In [2]:
import os
import time
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import dotenv
import pandas as pd
import requests

dotenv.load_dotenv()

LANGFUSE_PUBLIC_KEY = os.environ["LANGFUSE_PUBLIC_KEY"]
LANGFUSE_SECRET_KEY = os.environ["LANGFUSE_SECRET_KEY"]
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

TRACES_URL = f"{LANGFUSE_HOST}/api/public/traces"
SCORES_URL = f"{LANGFUSE_HOST}/api/public/v2/scores"

AUTH = (LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY)
session = requests.Session()


def iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def day_ranges(days_back: int, step_days: int = 1) -> list[tuple[str, str]]:
    today = datetime.now(timezone.utc).date()
    start = today - timedelta(days=days_back)

    ranges = []
    cur = start
    while cur <= today:
        end = min(cur + timedelta(days=step_days - 1), today)
        ranges.append(
            (
                iso_z(datetime.combine(cur, datetime.min.time(), tzinfo=timezone.utc)),
                iso_z(datetime.combine(end, datetime.max.time(), tzinfo=timezone.utc)),
            )
        )
        cur = end + timedelta(days=1)
    return ranges


def fetch_paginated(
    url: str,
    params: dict,
    *,
    limit: int = 100,
    max_items: int = 5000,
    max_retries: int = 5,
    timeout: int = 30,
) -> list[dict]:
    rows = []
    page = 1
    limit = min(max(1, limit), 100)

    while len(rows) < max_items:
        attempt = 0

        while True:
            try:
                r = session.get(
                    url,
                    auth=AUTH,
                    params={**params, "limit": limit, "page": page},
                    timeout=timeout,
                )

                if r.status_code == 429:
                    sleep_s = float(r.headers.get("Retry-After", "1"))
                    time.sleep(sleep_s)
                    continue

                if 500 <= r.status_code < 600:
                    if attempt >= max_retries:
                        r.raise_for_status()
                    time.sleep(min(2 ** attempt, 30))
                    attempt += 1
                    continue

                r.raise_for_status()
                payload = r.json()
                break

            except (requests.Timeout, requests.ConnectionError) as e:
                if attempt >= max_retries:
                    raise
                time.sleep(min(2 ** attempt, 30))
                attempt += 1

        chunk = payload.get("data") or []
        if not chunk:
            break

        rows.extend(chunk)

        meta = payload.get("meta") or {}
        current_page = meta.get("page", page)
        total_pages = meta.get("totalPages", page)

        if current_page >= total_pages:
            break

        page += 1

    return rows[:max_items]


def fetch_traces_slice(from_ts: str, to_ts: str, limit: int = 100) -> list[dict]:
    raw = fetch_paginated(
        TRACES_URL,
        {
            "tags": "batch_evaluation",
            "fromTimestamp": from_ts,
            "toTimestamp": to_ts,
        },
        limit=limit,
    )

    return [
        {
            "trace_id": t.get("id"),
            "trace_timestamp": t.get("timestamp") or t.get("createdAt"),
            "model": (t.get("metadata") or {}).get("model", "unknown"),
            "custom_id": (t.get("metadata") or {}).get("custom_id"),
            "batch_eval": bool((t.get("metadata") or {}).get("batch_eval")),
            "run_id": (t.get("metadata") or {}).get("run_id"),
        }
        for t in raw
    ]


def fetch_scores_slice(from_ts: str, to_ts: str, limit: int = 100) -> list[dict]:
    raw = fetch_paginated(
        SCORES_URL,
        {
            "name": "accuracy",
            "tags": "batch_evaluation",
            "fromTimestamp": from_ts,
            "toTimestamp": to_ts,
        },
        limit=limit,
    )

    return [
        {
            "score_id": s.get("id"),
            "trace_id": s.get("traceId"),
            "timestamp": s.get("timestamp") or s.get("createdAt"),
            "accuracy": s.get("value"),
        }
        for s in raw
    ]


def fetch_batch_traces(
    from_days: int = 7,
    slice_days: int = 1,
    per_request_limit: int = 100,
    max_workers: int = 2,
) -> pd.DataFrame:
    ranges = day_ranges(from_days, slice_days)

    trace_rows = []
    score_rows = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {}

        for start, end in ranges:
            futures[ex.submit(fetch_traces_slice, start, end, per_request_limit)] = "traces"
            futures[ex.submit(fetch_scores_slice, start, end, per_request_limit)] = "scores"

        for fut in as_completed(futures):
            kind = futures[fut]
            rows = fut.result()
            if kind == "traces":
                trace_rows.extend(rows)
            else:
                score_rows.extend(rows)

    traces_df = (
        pd.DataFrame(trace_rows).drop_duplicates(subset=["trace_id"])
        if trace_rows
        else pd.DataFrame(columns=["trace_id", "trace_timestamp", "model", "custom_id", "batch_eval", "run_id"])
    )

    scores_df = (
        pd.DataFrame(score_rows)
        if score_rows
        else pd.DataFrame(columns=["score_id", "trace_id", "timestamp", "accuracy"])
    )

    df = scores_df.merge(traces_df, on="trace_id", how="left")
    df["model"] = df["model"].fillna("unknown").astype(str)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["date"] = df["timestamp"].dt.date

    return df


df_traces = fetch_batch_traces(
    from_days=7,
    slice_days=1,
    per_request_limit=100,
    max_workers=10,
)

In [3]:
df_traces

,score_id,trace_id,timestamp,accuracy,trace_timestamp,model,custom_id,batch_eval,run_id,date
0,cc0bb90729297830,e5835dddbc1261de38dfa8b74b66eb8e,2026-03-09 19:27:28.697000+00:00,0,2026-03-09T19:27:28.696Z,gpt-4o-mini,3733929845806613018,True,20260309-165631Z,2026-03-09
1,08ba64c6377a15ff,b94cff2725b8769965c1e9e1f86e4dee,2026-03-09 19:27:28.696000+00:00,1,2026-03-09T19:27:28.695Z,gpt-4o-mini,3737642270557308674,True,20260309-165631Z,2026-03-09
2,453045dc5cac9c27,f7fb392816fdb6eae62b952a7320d32f,2026-03-09 19:27:28.695000+00:00,1,2026-03-09T19:27:28.694Z,gpt-4o-mini,7553611526704286998,True,20260309-165631Z,2026-03-09
3,8ef9fe694c488737,934b1441e7a0e7bf4f87e2d717e4e064,2026-03-09 19:27:28.694000+00:00,0,2026-03-09T19:27:28.693Z,gpt-4o-mini,7556881060701818134,True,20260309-165631Z,2026-03-09
4,f9b9105914f74ba9,db314c1c32c6b0aeddd11835f8f8aa20,2026-03-09 19:27:28.693000+00:00,1,2026-03-09T19:27:28.691Z,gpt-4o-mini,3730371735330494646,True,20260309-165631Z,2026-03-09
...,...,...,...,...,...,...,...,...,...,...
12615,43ce5731b14d1bc3,2ea5a8ff3c7d78a72a31c4137b7178ab,2026-03-10 06:54:18.355000+00:00,1,2026-03-10T06:54:18.354Z,gpt-5.4-2026-03-05,3733926866022333336,True,20260310-000055Z,2026-03-10
12616,212ac8cc2276bc6f,7e416bd9a35dc0b7e1f0967508114413,2026-03-10 06:54:18.354000+00:00,1,2026-03-10T06:54:18.353Z,gpt-5.4-2026-03-05,3737578749609921904,True,20260310-000055Z,2026-03-10
12617,d2b1a937256954d2,551821d3bf52dde790ea5bce591202f5,2026-03-10 06:54:18.353000+00:00,1,2026-03-10T06:54:18.351Z,gpt-5.4-2026-03-05,3731716582586975695,True,20260310-000055Z,2026-03-10
12618,9210df6c1c7b5c4b,359bcc29d8597f64a6ade1adf70878c1,2026-03-10 06:54:18.351000+00:00,1,2026-03-10T06:54:18.350Z,gpt-5.4-2026-03-05,3730374792961330564,True,20260310-000055Z,2026-03-10


In [5]:
# gpt 5 mini model
df_5mini = df_traces[df_traces["model"].str.contains("gpt-5-mini", case=False, na=False)]
# macnemar test of accuracy for gpt-5-mini trace timstamp 08.03 and 10.03

from datetime import date
import math

# select only the two dates of interest
_date_a = date(2026, 3, 8)
_date_b = date(2026, 3, 10)

_df_5mini_dates = df_5mini[df_5mini["timestamp"].dt.date.isin([_date_a, _date_b])].copy()
_df_5mini_dates["date"] = _df_5mini_dates["timestamp"].dt.date

# pivot to have one row per custom_id with accuracies for both days
_pivot = (
    _df_5mini_dates
    .pivot_table(index="custom_id", columns="date", values="accuracy")
    .dropna(subset=[_date_a, _date_b])
    .astype(int)
)

_y_a = _pivot[_date_a]
_y_b = _pivot[_date_b]

# contingency table counts
_a = ((_y_a == 1) & (_y_b == 1)).sum()
_b = ((_y_a == 1) & (_y_b == 0)).sum()
_c = ((_y_a == 0) & (_y_b == 1)).sum()
_d = ((_y_a == 0) & (_y_b == 0)).sum()

_n = _b + _c

# exact binomial version of McNemar's test (no external deps)
def _mcnemar_exact_pvalue(b: int, c: int) -> float:
    n = b + c
    if n == 0:
        return float("nan")
    k = min(b, c)
    p = 0.0
    for i in range(0, k + 1):
        p += math.comb(n, i)
    p = 2 * p / (2 ** n)
    return min(p, 1.0)

_mcnemar_p = _mcnemar_exact_pvalue(_b, _c)
_chi2 = ((abs(_b - _c) - 1) ** 2 / _n) if _n > 0 else float("nan")

print("McNemar contingency table (a, b; c, d):", [[_a, _b], [_c, _d]])
print("discordant pairs b, c:", _b, _c)
print("McNemar chi^2 (with continuity correction):", _chi2)
print("McNemar exact binomial p-value:", _mcnemar_p)


McNemar contingency table (a, b; c, d): [[np.int64(269), np.int64(13)], [np.int64(7), np.int64(11)]]
discordant pairs b, c: 13 7
McNemar chi^2 (with continuity correction): 1.25
McNemar exact binomial p-value: 0.26317596435546875


In [3]:
from pathlib import Path

# Inspect local CSV exports to find which traces lead to an 'unknown' model
TRACES_CSV = Path("../streamlit_app/langfuse_traces.csv")
SCORES_CSV = Path("../streamlit_app/langfuse_scores.csv")

traces_raw = pd.read_csv(TRACES_CSV)
scores_raw = pd.read_csv(SCORES_CSV)

# Normalise trace id column
traces_df = traces_raw.rename(columns={"id": "trace_id"})

# Decide which score trace-id column is used in the app
if "traceId" in scores_raw.columns:
    score_trace_col = "traceId"
elif "trace_id" in scores_raw.columns:
    score_trace_col = "trace_id"
else:
    score_trace_col = None

if score_trace_col is None:
    raise RuntimeError(
        "Scores CSV has no traceId / trace_id column; this is why app.py falls back to model='unknown'."
    )

# Build a merged view similar to streamlit_app.fetch_scores before the final cleanup
merged = scores_raw.merge(
    traces_df[["trace_id", "metadata.model", "metadata.custom_id", "metadata.batch_eval", "metadata.run_id"]],
    left_on=score_trace_col,
    right_on="trace_id",
    how="left",
)

# "Unknown" in the app comes from either missing trace rows or missing metadata.model
unknown_mask = merged["trace_id"].isna() | merged["metadata.model"].isna()

unknown_scores = merged.loc[unknown_mask].copy()

print(f"Total scores with unknown model after join: {len(unknown_scores)}")

# Show the problematic score rows with their trace ids and timestamps
cols = [score_trace_col, "timestamp", "id", "value"]
if "tags" in unknown_scores.columns:
    cols.append("tags")

display(unknown_scores[[c for c in cols if c in unknown_scores.columns]].head(20))

# Also show the corresponding raw trace rows (if any) for manual inspection
problem_trace_ids = unknown_scores[score_trace_col].dropna().unique()
display(traces_raw[traces_raw["id"].isin(problem_trace_ids)].head(20))

Total scores with unknown model after join: 15


,traceId,timestamp,id,value
303,2f56837c-d2bf-4e5e-9e71-51b9ee9a8e9f,2026-03-16T09:21:52.297Z,ac10b831-01a3-459b-b2b0-8e1e7376bed4,1.000000
309,2f56837c-d2bf-4e5e-9e71-51b9ee9a8e9f,2026-03-16T09:21:52.297Z,70934bcb-7d79-4ada-a1c7-8e840f2c2707,0.000000
314,2f56837c-d2bf-4e5e-9e71-51b9ee9a8e9f,2026-03-16T09:21:52.297Z,b8dc0fea-30ca-4f68-8e5e-5ddf6ae45854,0.029252
316,21518b71-900f-4709-a201-da7744701a74,2026-03-16T09:21:52.297Z,f62ed6f7-90e7-4f8f-baa7-4b1f363cd6e1,0.008436
317,21518b71-900f-4709-a201-da7744701a74,2026-03-16T09:21:52.297Z,1d16b3fe-2278-40f0-9e22-3a1e67e75d13,0.000000
320,21518b71-900f-4709-a201-da7744701a74,2026-03-16T09:21:52.297Z,9e0ca59e-35fd-49d5-8e41-9acb3d0dfdf5,1.000000
1200,c4c9d179-ec16-4603-aa09-4615539fac73,2026-03-16T09:21:28.628Z,1aa3865d-8af3-4b8d-b6cf-3a1c64c475b7,0.007021
1201,c4c9d179-ec16-4603-aa09-4615539fac73,2026-03-16T09:21:28.628Z,112ac0d2-bb34-4587-95cd-0b5503fd2d6a,1.000000
1204,c4c9d179-ec16-4603-aa09-4615539fac73,2026-03-16T09:21:28.628Z,d8a92d84-8c53-498d-9c58-21a86ac61817,0.000000
2111,14cf7d68-26e4-4303-9931-ff71cbcbdec6,2026-03-16T09:21:06.114Z,6a330e68-0ee7-458f-8e32-961989f852f0,1.000000


,id,timestamp,createdAt,tags,metadata.model,metadata.custom_id,metadata.batch_eval,metadata.run_id


In [4]:
from pathlib import Path
import pandas as pd

# Inspect local CSV exports to find which traces lead to an 'unknown' model
TRACES_CSV = Path("../streamlit_app/langfuse_traces.csv")
SCORES_CSV = Path("../streamlit_app/langfuse_scores.csv")

traces_raw = pd.read_csv(TRACES_CSV)
scores_raw = pd.read_csv(SCORES_CSV)

# Normalise trace id column
traces_df = traces_raw.rename(columns={"id": "trace_id"})

# Decide which score trace-id column is used in the app
if "traceId" in scores_raw.columns:
    score_trace_col = "traceId"
elif "trace_id" in scores_raw.columns:
    score_trace_col = "trace_id"
else:
    score_trace_col = None

if score_trace_col is None:
    raise RuntimeError("Scores CSV has no traceId / trace_id column; this is why app.py falls back to model='unknown'.")

# Build a merged view similar to streamlit_app.fetch_scores before the final cleanup
merged = scores_raw.merge(
    traces_df[["trace_id", "metadata.model", "metadata.custom_id", "metadata.batch_eval", "metadata.run_id"]],
    left_on=score_trace_col,
    right_on="trace_id",
    how="left",
)

# "Unknown" in the app comes from either missing trace rows or missing metadata.model
unknown_mask = merged["trace_id"].isna() | merged["metadata.model"].isna()

unknown_scores = merged.loc[unknown_mask].copy()

print(f"Total scores with unknown model after join: {len(unknown_scores)}")

# Show the problematic score rows with their trace ids and timestamps
unknown_scores[[score_trace_col, "timestamp", "id", "value", "tags"]].head(20)

# Also show the corresponding raw trace rows (if any) for manual inspection
problem_trace_ids = unknown_scores[score_trace_col].dropna().unique()
traces_raw[traces_raw["id"].isin(problem_trace_ids)].head(20)


Total scores with unknown model after join: 15


KeyError: "['tags'] not in index"